# BrownDye2 association-rate preparation

End-to-end preparation pipeline for a BrownDye2 simulation of a docked
complex (`complex.pdb`) approaching a protein substrate (`substrate.pdb`).
Everything except the long-running BrownDye trajectory step lives in this
notebook.

Steps:

1. Verify protonation and fix the complex protein chain (PropKa + PDBFixer).
2. Assign ligand bond orders from SMILES and write an SDF for antechamber.
3. Parameterise the complex with AmberTools/GAFF2 (`pdb4amber`, `obabel`,
   `antechamber`, `parmchk2`, `tleap`, ParmEd PQR export).
4. Solve APBS on the complex PQR.
5. Parameterise the protein-only substrate with PDB2PQR + PropKa.
6. Solve APBS on the substrate PQR.
7. Build BrownDye XML inputs (`pqr2xml`, contact types, `make_rxn_pairs`,
   `make_rxn_file`, `bd_top`).
8. Run BrownDye trajectories: `bash bdrun.sh` from a terminal (kept as a
   shell script because it can take hours).

Activate the AmberTools environment before launching Jupyter:

```bash
conda activate ambertools
```

In [ ]:
import os
import shutil
from pathlib import Path

from Bio.PDB import PDBIO, PDBParser
from rdkit import Chem

from mdpp.prep import ChainSelect, assign_topology, fix_pdb, run_propka

# Inputs (this directory).
EXAMPLE_DIR = Path.cwd()
COMPLEX_PDB = EXAMPLE_DIR / "complex.pdb"
SUBSTRATE_PDB = EXAMPLE_DIR / "substrate.pdb"

# Workspace layout: each stage owns a folder under tmp/, with transient
# tool files kept in <stage>/intermediate/.
TMP_ROOT = EXAMPLE_DIR / "tmp"
STEP_DIR = TMP_ROOT / "complex" / "prep"
INTERMEDIATE_DIR = STEP_DIR / "intermediate"
COMPLEX_AMBERTOOLS_DIR = TMP_ROOT / "complex" / "ambertools"
COMPLEX_AMBERTOOLS_INTERMEDIATE = COMPLEX_AMBERTOOLS_DIR / "intermediate"
COMPLEX_APBS_DIR = TMP_ROOT / "complex" / "apbs"
COMPLEX_APBS_INTERMEDIATE = COMPLEX_APBS_DIR / "intermediate"
SUBSTRATE_PDB2PQR_DIR = TMP_ROOT / "substrate" / "pdb2pqr"
SUBSTRATE_PDB2PQR_INTERMEDIATE = SUBSTRATE_PDB2PQR_DIR / "intermediate"
SUBSTRATE_APBS_DIR = TMP_ROOT / "substrate" / "apbs"
SUBSTRATE_APBS_INTERMEDIATE = SUBSTRATE_APBS_DIR / "intermediate"
BDPREP_DIR = TMP_ROOT / "bdprep"
BDPREP_INTERMEDIATE = BDPREP_DIR / "intermediate"
for path in (
    STEP_DIR,
    INTERMEDIATE_DIR,
    COMPLEX_AMBERTOOLS_DIR,
    COMPLEX_AMBERTOOLS_INTERMEDIATE,
    COMPLEX_APBS_DIR,
    COMPLEX_APBS_INTERMEDIATE,
    SUBSTRATE_PDB2PQR_DIR,
    SUBSTRATE_PDB2PQR_INTERMEDIATE,
    SUBSTRATE_APBS_DIR,
    SUBSTRATE_APBS_INTERMEDIATE,
    BDPREP_DIR,
    BDPREP_INTERMEDIATE,
):
    path.mkdir(parents=True, exist_ok=True)

# Ligand template (used to assign bond orders to the PDB ligand block).
LIGAND_SMILES = (
    r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)"
    r"[C@@H](O)[C@H]3O"
)
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

# AmberTools (complex parameterisation).
PROTEIN_FF = "leaprc.protein.ff19SB"
LIGAND_FF = "leaprc.gaff2"
PB_RADII = "mbondi3"
STRIP_PROTEIN_H = True

# PDB2PQR (substrate parameterisation).
PDB2PQR_FF = "AMBER"

# APBS grid + solvent parameters (used by both complex and substrate).
IONIC_STRENGTH = 0.150
SOLUTE_DIELECTRIC = 2.0
SOLVENT_DIELECTRIC = 78.54
SOLVENT_RADIUS = 1.4
TEMPERATURE = 298.15
FINE_SPACING = 0.75
FINE_PADDING = 60.0
COARSE_PADDING = 100.0

# BrownDye install + bodies.
BD_HOME = Path(os.environ.get("BD_HOME", "/apps/browndye2"))
BD_BIN = Path(os.environ.get("BD_BIN", str(BD_HOME / "bin")))
BD_AUX = Path(os.environ.get("BD_AUX", str(BD_HOME / "aux")))
CORE0 = "complex"
CORE1 = "substrate"
CORE0_IS_PROTEIN = True
CORE1_IS_PROTEIN = True
CORE0_DIELECTRIC = 4.0
CORE1_DIELECTRIC = 4.0

# BrownDye reaction criteria + run parameters consumed by bdrun.sh.
RXN_SEARCH_DISTANCE = 5.5
RXN_DISTANCE = 5.5
RXN_NEEDED = 3
N_THREADS = 1
SEED = 11111111
N_TRAJECTORIES = 1000
N_TRAJECTORIES_PER_OUTPUT = 10
MAX_N_STEPS = 1_000_000
RESULTS_FILE = "results.xml"
BD_SOLVENT_DIELECTRIC = 78.0  # BrownDye uses kT-units; differs from APBS sdie
RELATIVE_VISCOSITY = 1.0
KT = 1.0
DESOLVATION_PARAMETER = 1.0
DEBYE_LENGTH: float | None = None  # inferred from APBS logs in Step 7

# Export every shell-facing knob so subsequent %%bash cells inherit the
# same configuration without having to re-type defaults.
os.environ.update({
    "TMP_ROOT": str(TMP_ROOT),
    "COMPLEX_AMBERTOOLS_DIR": str(COMPLEX_AMBERTOOLS_DIR),
    "COMPLEX_AMBERTOOLS_INTERMEDIATE": str(COMPLEX_AMBERTOOLS_INTERMEDIATE),
    "COMPLEX_APBS_DIR": str(COMPLEX_APBS_DIR),
    "COMPLEX_APBS_INTERMEDIATE": str(COMPLEX_APBS_INTERMEDIATE),
    "SUBSTRATE_PDB2PQR_DIR": str(SUBSTRATE_PDB2PQR_DIR),
    "SUBSTRATE_PDB2PQR_INTERMEDIATE": str(SUBSTRATE_PDB2PQR_INTERMEDIATE),
    "SUBSTRATE_APBS_DIR": str(SUBSTRATE_APBS_DIR),
    "SUBSTRATE_APBS_INTERMEDIATE": str(SUBSTRATE_APBS_INTERMEDIATE),
    "BDPREP_DIR": str(BDPREP_DIR),
    "BDPREP_INTERMEDIATE": str(BDPREP_INTERMEDIATE),
    "SUBSTRATE_PDB": str(SUBSTRATE_PDB),
    "PROTEIN_FF": PROTEIN_FF,
    "LIGAND_FF": LIGAND_FF,
    "PB_RADII": PB_RADII,
    "STRIP_PROTEIN_H": "1" if STRIP_PROTEIN_H else "0",
    "PDB2PQR_FF": PDB2PQR_FF,
    "PH": f"{PH}",
    "BD_HOME": str(BD_HOME),
    "BD_BIN": str(BD_BIN),
    "BD_AUX": str(BD_AUX),
    "CORE0": CORE0,
    "CORE1": CORE1,
    "RXN_SEARCH_DISTANCE": f"{RXN_SEARCH_DISTANCE}",
    "RXN_DISTANCE": f"{RXN_DISTANCE}",
    "RXN_NEEDED": f"{RXN_NEEDED}",
})
os.environ["PATH"] = f"{BD_BIN}:{BD_AUX}:{os.environ.get('PATH', '')}"

In [ ]:
import re
from math import ceil


def write_apbs_input(stem: str, work_dir: Path) -> Path:
    """Write an APBS input file `{stem}.in` for `{stem}.pqr` in `work_dir`.

    The grid is centred on the radius-inflated atom bounding box and padded
    by ``FINE_PADDING`` / ``COARSE_PADDING``. Grid dimensions are rounded up
    to the next ``c * 2**n + 1`` value required by APBS multigrid.

    Args:
        stem: PQR file stem (without extension) inside ``work_dir``.
        work_dir: directory containing ``{stem}.pqr``; receives ``{stem}.in``.

    Returns:
        Path to the written APBS input file.
    """
    pqr_path = work_dir / f"{stem}.pqr"
    apbs_input = work_dir / f"{stem}.in"

    coords: list[tuple[float, float, float]] = []
    radii: list[float] = []
    with pqr_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            fields = line.split()
            if len(fields) < 10:
                continue
            x, y, z = (float(value) for value in fields[-5:-2])
            radius = float(fields[-1])
            coords.append((x, y, z))
            radii.append(radius)
    if not coords:
        raise ValueError(f"No atoms found in {pqr_path}")

    lower = [min(coord[i] - radii[index] for index, coord in enumerate(coords)) for i in range(3)]
    upper = [max(coord[i] + radii[index] for index, coord in enumerate(coords)) for i in range(3)]
    center = [(low + high) / 2.0 for low, high in zip(lower, upper, strict=True)]
    span = [high - low for low, high in zip(lower, upper, strict=True)]
    fglen = [value + FINE_PADDING for value in span]
    cglen = [max(value + COARSE_PADDING, fine) for value, fine in zip(span, fglen, strict=True)]

    candidates = sorted({c * 2**n + 1 for c in range(1, 7) for n in range(1, 12)})

    def apbs_friendly_dime(length: float) -> int:
        target = ceil(length / FINE_SPACING) + 1
        for candidate in candidates:
            if candidate >= target:
                return candidate
        return candidates[-1]

    dime = [apbs_friendly_dime(length) for length in fglen]

    apbs_input.write_text(
        f"""read
    mol pqr {stem}.pqr
end
elec
    mg-auto
    dime {dime[0]} {dime[1]} {dime[2]}
    cglen {cglen[0]:.4f} {cglen[1]:.4f} {cglen[2]:.4f}
    fglen {fglen[0]:.4f} {fglen[1]:.4f} {fglen[2]:.4f}
    cgcent {center[0]:.4f} {center[1]:.4f} {center[2]:.4f}
    fgcent {center[0]:.4f} {center[1]:.4f} {center[2]:.4f}
    mol 1
    lpbe
    bcfl sdh
    ion charge -1.00 conc {IONIC_STRENGTH:.4f} radius 1.8150
    ion charge 1.00 conc {IONIC_STRENGTH:.4f} radius 1.8750
    pdie {SOLUTE_DIELECTRIC:.4f}
    sdie {SOLVENT_DIELECTRIC:.4f}
    srfm smol
    chgm spl2
    sdens 10.00
    srad {SOLVENT_RADIUS:.4f}
    swin 0.30
    temp {TEMPERATURE:.2f}
    calcenergy total
    calcforce no
    write pot dx {stem}
end
print elecEnergy 1 end
quit
""",
        encoding="utf-8",
    )
    return apbs_input


def infer_debye_length(*apbs_logs: Path) -> float:
    """Return the first Debye length parsed from any of the given APBS logs."""
    for path in apbs_logs:
        if not path.is_file():
            continue
        text = path.read_text(errors="ignore")
        match = re.search(r"[Dd]ebye[- ]length[:\s]+([0-9.]+)", text)
        match = match or re.search(r"[Gg]ot debye length\s+([0-9.]+)", text)
        if match:
            return float(match.group(1))
    raise RuntimeError(f"Could not infer Debye length from any of: {[str(p) for p in apbs_logs]}")


def write_contact_types(mol0_pqr: Path, mol1_pqr: Path, out_path: Path) -> None:
    """Write BrownDye `contact_types.xml` with unique heavy-atom keys per body."""

    def heavy_atom_keys(path: Path) -> list[tuple[str, str]]:
        seen: set[tuple[str, str]] = set()
        ordered: list[tuple[str, str]] = []
        for line in path.read_text().splitlines():
            if not line.startswith(("ATOM", "HETATM")):
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            atom = parts[2]
            residue = parts[3]
            key = (atom, residue)
            if atom and not atom.upper().startswith("H") and key not in seen:
                seen.add(key)
                ordered.append(key)
        return ordered

    lines: list[str] = ["<contacts>", "  <combinations>"]
    for label, entries in (
        ("molecule0", heavy_atom_keys(mol0_pqr)),
        ("molecule1", heavy_atom_keys(mol1_pqr)),
    ):
        lines.append(f"    <{label}>")
        for atom, residue in entries:
            lines.append(
                f"      <contact> <atom> {atom} </atom> <residue> {residue} </residue> </contact>"
            )
        lines.append(f"    </{label}>")
    lines += ["  </combinations>", "</contacts>", ""]
    out_path.write_text("\n".join(lines), encoding="utf-8")

## Step 1: Verify protonation and fix protein

Extract the protein chain, run PropKa as an explicit pKa/protonation-state check at the target pH, then add missing residues/atoms/hydrogens via PDBFixer.

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]
chains = {chain.id: chain for chain in model}

pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract protein chain into this step's intermediate folder.
protein_pdb = INTERMEDIATE_DIR / "protein.pdb"
pdb_io.save(str(protein_pdb), ChainSelect(PROTEIN_CHAIN_ID))

# Verify protonation with PropKa before PDBFixer assigns hydrogens.
propka_result = run_propka(protein_pdb)
nonstandard = propka_result.get_nonstandard(PH)
propka_report = STEP_DIR / "protein_propka.tsv"
with propka_report.open("w") as f:
    f.write(
        "residue_type\tres_num\tchain_id\tpka\tmodel_pka\tpropka_protonated\tmodel_protonated\n"
    )
    for residue in propka_result.residues:
        f.write(
            f"{residue.residue_type}\t{residue.res_num}\t{residue.chain_id}\t"
            f"{residue.pka:.3f}\t{residue.model_pka:.3f}\t"
            f"{residue.is_protonated_at(PH)}\t{residue.is_default_protonated_at(PH)}\n"
        )

print(f"PropKa residues checked: {len(propka_result.residues)} -> {propka_report}")
if nonstandard:
    print(f"PropKa predicts {len(nonstandard)} non-standard protonation state(s) at pH {PH}:")
    for residue in nonstandard:
        print(
            f"  {residue.label}: pKa={residue.pka:.2f}, model={residue.model_pka:.2f}, "
            f"PropKa={'protonated' if residue.is_protonated_at(PH) else 'deprotonated'}, "
            f"model={'protonated' if residue.is_default_protonated_at(PH) else 'deprotonated'}"
        )
else:
    print(f"PropKa agrees with model-pKa defaults for all titratable residues at pH {PH}.")

protein_fixed_pdb = STEP_DIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 2: Assign ligand topology

Extract ligand chain, assign bond orders from a SMILES template, and write an SDF
for antechamber. The SMILES is needed because PDB files lack bond-order information.

In [ ]:
# Extract ligand chain to the step intermediate folder for RDKit parsing.
ligand_pdb = INTERMEDIATE_DIR / "ligand.pdb"
pdb_io.save(str(ligand_pdb), ChainSelect(LIGAND_CHAIN_ID))

# Assign bond orders from SMILES template.
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"Ligand net charge: {ligand_net_charge}")

mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
mol_assigned = assign_topology(mol, template_mol)

# Write SDF for antechamber and metadata for the later AmberTools cells.
lig_resnames = sorted({res.get_resname().strip() for res in chains[LIGAND_CHAIN_ID].get_residues()})
lig_resname = lig_resnames[0]
mol_assigned.SetProp("_Name", lig_resname)

ligand_sdf = STEP_DIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)

(STEP_DIR / "ligand_charge.txt").write_text(f"{ligand_net_charge}\n")
(STEP_DIR / "ligand_resname.txt").write_text(f"{lig_resname}\n")

# Make resname + net charge available to subsequent bash cells.
os.environ["LIG_RESNAME"] = lig_resname
os.environ["NET_CHARGE"] = str(ligand_net_charge)
print(f"Ligand SDF -> {ligand_sdf}")
print(f"Ligand metadata -> {STEP_DIR / 'ligand_charge.txt'}, {STEP_DIR / 'ligand_resname.txt'}")

## Step 3: Parameterise the complex with AmberTools

Run the standard AmberTools pipeline on the prepared protein and ligand:

1. `pdb4amber` cleans the protein PDB for tleap.
2. `obabel` converts the ligand SDF to mol2 as an antechamber seed; we then
   patch the residue name to match the ligand chain.
3. `antechamber` assigns AM1-BCC charges and GAFF2 atom types.
4. `parmchk2` generates any missing GAFF2 parameters.
5. `tleap` combines the protein (ff19SB) and ligand (GAFF2) into a complex
   topology and saves prmtop/rst7 for protein, ligand, and complex bodies.
6. ParmEd exports each topology to PQR (charge + radius per atom).

Outputs live in `$COMPLEX_AMBERTOOLS_DIR`; transient files in its
`intermediate/` folder.

In [ ]:
# Stage the Step 1/2 outputs into the AmberTools intermediate folder.
shutil.copy(STEP_DIR / "protein_fixed.pdb", COMPLEX_AMBERTOOLS_INTERMEDIATE / "protein_fixed.pdb")
shutil.copy(STEP_DIR / "ligand.sdf", COMPLEX_AMBERTOOLS_INTERMEDIATE / "ligand.sdf")

In [ ]:
%%bash
set -euo pipefail
cd "$COMPLEX_AMBERTOOLS_INTERMEDIATE"

echo "=== 1. pdb4amber ==="
pdb4amber_args=(-i protein_fixed.pdb -o protein_amber.pdb -d --no-conect)
if [[ "$STRIP_PROTEIN_H" == "1" ]]; then
    pdb4amber_args+=(-y)
fi
pdb4amber "${pdb4amber_args[@]}"

echo "=== 2a. obabel: SDF -> mol2 ==="
obabel ligand.sdf -O ligand_seed.mol2

In [ ]:
# Patch the residue name written by obabel so it matches the docked ligand.
ligand_seed_mol2 = COMPLEX_AMBERTOOLS_INTERMEDIATE / "ligand_seed.mol2"
text = ligand_seed_mol2.read_text()
for old in ("UNL1", "UNL", "UNK"):
    text = text.replace(old, lig_resname)
if lig_resname not in text:
    raise RuntimeError(
        f"Residue name {lig_resname!r} not found in {ligand_seed_mol2} after replacement"
    )
ligand_seed_mol2.write_text(text)
print(f"Patched residue name -> {lig_resname}")

In [ ]:
%%bash
set -euo pipefail
cd "$COMPLEX_AMBERTOOLS_INTERMEDIATE"

echo "=== 2b. antechamber (AM1-BCC + GAFF2) ==="
antechamber \
    -i ligand_seed.mol2 -fi mol2 \
    -o ligand_amber.mol2 -fo mol2 \
    -c bcc -s 2 -at gaff2 \
    -nc "$NET_CHARGE" -rn "$LIG_RESNAME"

echo "=== 2c. parmchk2 (missing GAFF2 parameters) ==="
parmchk2 -i ligand_amber.mol2 -f mol2 -o ligand.frcmod

echo "=== 3. tleap (ff19SB protein + GAFF2 ligand -> complex) ==="
cat >tleap.in <<EOF
source $PROTEIN_FF
source $LIGAND_FF

$LIG_RESNAME = loadmol2 ligand_amber.mol2
loadamberparams ligand.frcmod
protein = loadpdb protein_amber.pdb
complex = combine {protein $LIG_RESNAME}

set default PBRadii $PB_RADII
saveamberparm protein protein.prmtop protein.rst7
saveamberparm $LIG_RESNAME ligand.prmtop ligand.rst7
saveamberparm complex complex.prmtop complex.rst7
quit
EOF
tleap -f tleap.in

In [ ]:
# ParmEd: convert each topology/coord pair to PQR (charge + radius per atom).
import parmed as pmd

for stem in ("protein", "ligand", "complex"):
    parm = pmd.load_file(
        str(COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.prmtop"),
        xyz=str(COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.rst7"),
    )
    parm.save(str(COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.pqr"), overwrite=True)
    for suffix in ("prmtop", "rst7", "pqr"):
        shutil.copy(
            COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.{suffix}",
            COMPLEX_AMBERTOOLS_DIR / f"{stem}.{suffix}",
        )

print("Complex AmberTools outputs:")
for stem in ("protein", "ligand", "complex"):
    print("  ", COMPLEX_AMBERTOOLS_DIR / f"{stem}.pqr")

## Step 4: Solve APBS for the complex

Generate an APBS input file (`complex.in`) for the complex PQR and run
`apbs` to produce the electrostatic potential map `complex.dx`. The grid
is sized from the radius-inflated atom bounding box of `complex.pqr` and
padded by `FINE_PADDING` / `COARSE_PADDING` (set in the globals cell).

Protein-only and ligand-only diagnostic maps are optional - rerun
`write_apbs_input("protein", ...)` / `write_apbs_input("ligand", ...)` and
the bash cell with a different `STEMS=` if you want them.

In [ ]:
# Stage complex.pqr into the APBS intermediate folder and write complex.in.
shutil.copy(
    COMPLEX_AMBERTOOLS_DIR / "complex.pqr",
    COMPLEX_APBS_INTERMEDIATE / "complex.pqr",
)
apbs_in = write_apbs_input("complex", COMPLEX_APBS_INTERMEDIATE)
print(f"APBS input -> {apbs_in}")

# Optional diagnostics (uncomment to also map protein and ligand alone):
# for diag in ("protein", "ligand"):
#     shutil.copy(COMPLEX_AMBERTOOLS_DIR / f"{diag}.pqr", COMPLEX_APBS_INTERMEDIATE / f"{diag}.pqr")
#     write_apbs_input(diag, COMPLEX_APBS_INTERMEDIATE)

os.environ["STEMS"] = "complex"

In [ ]:
%%bash
set -euo pipefail
cd "$COMPLEX_APBS_INTERMEDIATE"

for stem in $STEMS; do
    echo "=== APBS: $stem ==="
    apbs "$stem.in" 2>&1 | tee "$stem.apbs.log"

    if [[ -s "$stem-PE0.dx" ]]; then
        mv "$stem-PE0.dx" "$stem.dx"
    elif [[ -s "$stem.pqr.dx" ]]; then
        mv "$stem.pqr.dx" "$stem.dx"
    fi

    if [[ ! -s "$stem.dx" ]]; then
        echo "Missing $stem.dx" >&2
        exit 1
    fi
    cp "$stem.in" "$COMPLEX_APBS_DIR/$stem.in"
    cp "$stem.apbs.log" "$COMPLEX_APBS_DIR/$stem.apbs.log"
    cp "$stem.dx" "$COMPLEX_APBS_DIR/$stem.dx"
done

ls -lh "$COMPLEX_APBS_DIR"/*.dx

## Step 5: Parameterise the substrate with PDB2PQR

PDB2PQR 3.7.1 does not expose an ff19SB option, so the substrate uses
PDB2PQR's bundled AMBER force field together with PropKa for pH-dependent
titration. The complex body still uses AmberTools/tleap ff19SB.

In [ ]:
%%bash
set -euo pipefail
cp "$SUBSTRATE_PDB" "$SUBSTRATE_PDB2PQR_INTERMEDIATE/substrate.pdb"
cd "$SUBSTRATE_PDB2PQR_INTERMEDIATE"

echo "=== PDB2PQR substrate ==="
pdb2pqr \
    --ff "$PDB2PQR_FF" \
    --ffout "$PDB2PQR_FF" \
    --keep-chain \
    --drop-water \
    --titration-state-method propka \
    --with-ph "$PH" \
    substrate.pdb \
    substrate.pqr 2>&1 | tee substrate.pdb2pqr.log

cp substrate.pqr "$SUBSTRATE_PDB2PQR_DIR/substrate.pqr"
cp substrate.pdb2pqr.log "$SUBSTRATE_PDB2PQR_DIR/substrate.pdb2pqr.log"
ls -lh "$SUBSTRATE_PDB2PQR_DIR/substrate.pqr"

## Step 6: Solve APBS for the substrate

Same flow as Step 4, applied to the substrate PQR.

In [ ]:
shutil.copy(
    SUBSTRATE_PDB2PQR_DIR / "substrate.pqr",
    SUBSTRATE_APBS_INTERMEDIATE / "substrate.pqr",
)
apbs_in = write_apbs_input("substrate", SUBSTRATE_APBS_INTERMEDIATE)
print(f"APBS input -> {apbs_in}")

In [ ]:
%%bash
set -euo pipefail
cd "$SUBSTRATE_APBS_INTERMEDIATE"

stem=substrate
echo "=== APBS: $stem ==="
apbs "$stem.in" 2>&1 | tee "$stem.apbs.log"

if [[ -s "$stem-PE0.dx" ]]; then
    mv "$stem-PE0.dx" "$stem.dx"
elif [[ -s "$stem.pqr.dx" ]]; then
    mv "$stem.pqr.dx" "$stem.dx"
fi

if [[ ! -s "$stem.dx" ]]; then
    echo "Missing $stem.dx" >&2
    exit 1
fi
cp "$stem.in" "$SUBSTRATE_APBS_DIR/$stem.in"
cp "$stem.apbs.log" "$SUBSTRATE_APBS_DIR/$stem.apbs.log"
cp "$stem.dx" "$SUBSTRATE_APBS_DIR/$stem.dx"
ls -lh "$SUBSTRATE_APBS_DIR/$stem.dx"

## Step 7: Build BrownDye XML inputs

Stage the complex and substrate PQR/DX files into `$BDPREP_INTERMEDIATE`,
infer the Debye length from the APBS logs, then walk the BrownDye setup
pipeline:

1. `pqr2xml` converts each PQR to BrownDye `atoms.xml`.
2. `contact_types.xml` lists every unique heavy-atom name per body.
3. `make_rxn_pairs` finds bound-pose contact pairs within `RXN_SEARCH_DISTANCE`.
4. `make_rxn_file` turns them into a reaction definition that fires when
   `RXN_NEEDED` pairs come within `RXN_DISTANCE` of each other.
5. `input.xml` is the BrownDye top-level config consumed by `bd_top`,
   which validates and writes `${CORE0}_${CORE1}_simulation.xml`.

`bdrun.sh` (kept as a shell script for the long-running trajectory step)
reads `$BDPREP_INTERMEDIATE/${CORE0}_${CORE1}_simulation.xml` directly.

In [ ]:
# Stage the four BrownDye inputs.
shutil.copy(COMPLEX_AMBERTOOLS_DIR / f"{CORE0}.pqr", BDPREP_INTERMEDIATE / f"{CORE0}.pqr")
shutil.copy(SUBSTRATE_PDB2PQR_DIR / f"{CORE1}.pqr", BDPREP_INTERMEDIATE / f"{CORE1}.pqr")
shutil.copy(COMPLEX_APBS_DIR / f"{CORE0}.dx", BDPREP_INTERMEDIATE / f"{CORE0}.dx")
shutil.copy(SUBSTRATE_APBS_DIR / f"{CORE1}.dx", BDPREP_INTERMEDIATE / f"{CORE1}.dx")

# Infer Debye length from the APBS logs unless the user pinned it manually.
if DEBYE_LENGTH is None:
    DEBYE_LENGTH = infer_debye_length(
        COMPLEX_APBS_DIR / f"{CORE0}.apbs.log",
        SUBSTRATE_APBS_DIR / f"{CORE1}.apbs.log",
    )
os.environ["DEBYE_LENGTH"] = f"{DEBYE_LENGTH}"
print(f"Debye length: {DEBYE_LENGTH} A")

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== 1. PQR -> BrownDye XML ==="
pqr2xml <"${CORE0}.pqr" >"${CORE0}_atoms.xml"
pqr2xml <"${CORE1}.pqr" >"${CORE1}_atoms.xml"

In [ ]:
# Heavy-atom contact criteria for make_rxn_pairs.
write_contact_types(
    BDPREP_INTERMEDIATE / f"{CORE0}.pqr",
    BDPREP_INTERMEDIATE / f"{CORE1}.pqr",
    BDPREP_INTERMEDIATE / "contact_types.xml",
)
print("contact_types.xml ->", BDPREP_INTERMEDIATE / "contact_types.xml")

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== 2. Reaction criteria ==="
make_rxn_pairs -nonred \
    -mol0 "${CORE0}_atoms.xml" \
    -mol1 "${CORE1}_atoms.xml" \
    -ctypes contact_types.xml \
    -dist "$RXN_SEARCH_DISTANCE" >reaction_pairs.xml
make_rxn_file \
    -pairs reaction_pairs.xml \
    -state_from before \
    -state_to after \
    -rxn association \
    -mol0 "$CORE0" "$CORE0" \
    -mol1 "$CORE1" "$CORE1" \
    -distance "$RXN_DISTANCE" \
    -nneeded "$RXN_NEEDED" >reactions.xml

In [ ]:
# BrownDye top-level input consumed by bd_top.
input_xml = BDPREP_INTERMEDIATE / "input.xml"
input_xml.write_text(
    f"""<top>
  <n_threads> {N_THREADS} </n_threads>
  <seed> {SEED} </seed>
  <output> {RESULTS_FILE} </output>

  <n_trajectories> {N_TRAJECTORIES} </n_trajectories>
  <n_trajectories_per_output> {N_TRAJECTORIES_PER_OUTPUT} </n_trajectories_per_output>
  <max_n_steps> {MAX_N_STEPS} </max_n_steps>

  <system>
    <reaction_file> reactions.xml </reaction_file>

    <solvent>
      <debye_length> {DEBYE_LENGTH} </debye_length>
      <dielectric> {BD_SOLVENT_DIELECTRIC} </dielectric>
      <relative_viscosity> {RELATIVE_VISCOSITY} </relative_viscosity>
      <kT> {KT} </kT>
      <desolvation_parameter> {DESOLVATION_PARAMETER} </desolvation_parameter>
      <solvent_radius> {SOLVENT_RADIUS} </solvent_radius>
    </solvent>

    <time_step_tolerances>
      <minimum_core_dt> 0.0 </minimum_core_dt>
      <minimum_core_reaction_dt> 0.0 </minimum_core_reaction_dt>
    </time_step_tolerances>

    <group>
      <name> {CORE0} </name>
      <core>
        <name> {CORE0} </name>
        <all_in_surface> false </all_in_surface>
        <is_protein> {str(CORE0_IS_PROTEIN).lower()} </is_protein>
        <atoms> {CORE0}_atoms.xml </atoms>
        <electric_field>
          <grid> {CORE0}.dx </grid>
        </electric_field>
        <dielectric> {CORE0_DIELECTRIC} </dielectric>
      </core>
    </group>

    <group>
      <name> {CORE1} </name>
      <core>
        <name> {CORE1} </name>
        <all_in_surface> false </all_in_surface>
        <is_protein> {str(CORE1_IS_PROTEIN).lower()} </is_protein>
        <atoms> {CORE1}_atoms.xml </atoms>
        <electric_field>
          <grid> {CORE1}.dx </grid>
        </electric_field>
        <dielectric> {CORE1_DIELECTRIC} </dielectric>
      </core>
    </group>
  </system>
</top>
""",
    encoding="utf-8",
)
print("input.xml ->", input_xml)

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== 3. BrownDye top-level input ==="
bd_top input.xml

for file in \
    "${CORE0}_atoms.xml" \
    "${CORE1}_atoms.xml" \
    contact_types.xml \
    reaction_pairs.xml \
    reactions.xml \
    input.xml \
    "${CORE0}_${CORE1}_simulation.xml"; do
    cp "$file" "$BDPREP_DIR/$file"
done

echo "Simulation XML: $BDPREP_DIR/${CORE0}_${CORE1}_simulation.xml"

## Step 8: Run BrownDye trajectories

Trajectory propagation is the only long-running step (`N_TRAJECTORIES=1000`
by default can take hours, weighted-ensemble mode more), so it is **kept as
a shell script**. From a terminal in this directory:

```bash
conda activate ambertools
cd examples/browndye
bash bdrun.sh            # standard NAM mode
# or
MODE=we bash bdrun.sh    # weighted-ensemble mode
```

`bdrun.sh` reads `tmp/bdprep/intermediate/${CORE0}_${CORE1}_simulation.xml`
(populated by Step 7), runs `nam_simulation` or `we_simulation`, and writes
`tmp/bdrun/results.xml` plus `tmp/bdrun/rate_constant.txt`.

For visualisation after the APBS step completes, see `viz_complex_apbs.pml`
and `viz_complex_apbs.cxc` (PyMOL / ChimeraX).